# SDP pipeline analisis de log

Cada SDP guardar eventos y metricas en la localización del almacenamiento definido para el pipeline. Con esta tabla podemos ver que está pasando y la calidad de datos procesados.

Puede consultas las tablas en forma directa con SQL y enviar alertas. 
Este notebook extrae y analiza metricas de espectativas para construir KPI.

Publicar el log de eventos en el UC:
1.	Settings > “Advanced settings” > presione el botón “Edit advanced settings”.
2.	Active la cada de chequeo “Publish evento log to unity catalog”.
3.	Suministre el nombre de catálogo, esquema para log de eventos y nombre de la tabla, por ejemplo, event_log_MiPipeline.

## Consultar SDP eventos en UC

In [0]:
%sql
SELECT * FROM medallion_dev.bronze.event_log_medallon_pipeline

## Analizar estructura de la tabla event log

La columna `details` contiene metadatos de cada evento. Los campos dependen del tipo de evento:
* `user_action` Acciones de usuario como crear pipeline
* `flow_definition` Cuando despliega o modifica pipeline y tiene lineage, schema y plan de ejecución.
  * `output_dataset` y `input_datasets` - output table/view and its upstream table(s)/view(s)
  * `flow_type` - whether this is a complete or append flow
  * `explain_text` - the Spark explain plan
* `flow_progress` Events occur when a data flow starts running or finishes processing a batch of data
  * `metrics` - currently contains `num_output_rows`
  * `data_quality` - contains an array of the results of the data quality rules for this particular dataset
    * `dropped_records`
    * `expectations`
      * `name`, `dataset`, `passed_records`, `failed_records`

In [0]:
%sql
SELECT
  details:flow_definition.output_dataset,
  details:flow_definition.input_datasets,
  details:flow_definition.flow_type,
  details:flow_definition.schema,
  details:flow_definition
FROM medallion_dev.bronze.event_log_medallon_pipeline
WHERE details:flow_definition IS NOT NULL
ORDER BY timestamp
LIMIT 10

In [0]:
%sql
SELECT
  id,
  expectations.dataset,
  expectations.name,
  expectations.failed_records,
  expectations.passed_records
FROM(
  SELECT 
    id,
    timestamp,
    details:flow_progress.metrics,
    details:flow_progress.data_quality.dropped_records,
    explode(from_json(details:flow_progress:data_quality:expectations
             ,schema_of_json("[{'name':'str', 'dataset':'str', 'passed_records':42, 'failed_records':42}]"))) expectations
  FROM medallion_dev.bronze.event_log_medallon_pipeline
  WHERE details:flow_progress.metrics IS NOT NULL) data_quality

## Eso es todo! Las metricas de calidad están listas! 

La tabla puede ser consultada usando DBSQL. Abra el <a dbdemos-dashboard-id="sdp-quality-stat" href='/sql/dashboardsv3/01f118a0ddea16c09a3d5954c9c49066' target="_blank">Dashboard Log SDP calidad de datos</a>

Inicio [Ir a tras a README.md]($../README.md)